# W7D1 — Chunking, and What It Decides — Lab

**Week 7 · Day 1 · Retrieval, RAG and Recommenders** · Lab

This morning a model answered a question about a company it had never heard of, confidently and
wrongly, and the fix was to stop asking it to remember and start handing it documents. Today you
build the half of that system nobody looks at when it breaks: **the cut**.

No language model appears in this notebook, deliberately. Retrieval is the component you will
misdiagnose all week, so day one isolates it completely — every failure you see here happened
before any generator ran.

You start from the 120-word passage on this morning's slide and reproduce its numbers exactly,
including the retrieved chunk that opens `days of purchase.` with the number missing. Then you cut
24 real documents four ways, measure **context recall** for 40 questions against every strategy,
and read a table that says plainly that no strategy wins everything.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٧ اليوم ١ — التقطيع وما يقرّره

**الأسبوع السابع · اليوم الأول · الاسترجاع والتوليد المعزّز والتوصية** · معمل

هذا الصباح أجاب نموذج عن سؤال يخصّ شركة لم يسمع بها قط، بثقة وبخطأ، وكان العلاج أن نكفّ عن مطالبته
بالتذكّر وأن نناوله الوثائق. واليوم تبني النصف الذي لا ينظر إليه أحد حين يتعطّل النظام: **القطع**.

ولا يظهر أي نموذج لغوي في هذا الدفتر، وذلك عن قصد. فالاسترجاع هو المكوّن الذي ستُخطئ تشخيصه طوال
الأسبوع، فيعزله اليوم الأول عزلًا تامًّا — وكل إخفاق تراه هنا وقع قبل أن يعمل أي مُولِّد.

تبدأ من فقرة المئة وعشرين كلمة التي في شريحة الصباح وتُعيد أرقامها بالضبط، ومنها المقطع المسترجَع
الذي يبدأ بـ`days of purchase.` والرقم غائب. ثم تقطّع أربعًا وعشرين وثيقة حقيقية بأربع طرق، وتقيس
**استدعاء السياق** لأربعين سؤالًا عند كل طريقة، وتقرأ جدولًا يقول صراحةً إن لا طريقة تربح كل شيء.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Cut a document by fixed size, by sentence, and by topic, and say what each cut costs.
- Explain what overlap buys and what you pay for it, with a number rather than an opinion.
- Keep source, section and chunk index on every chunk, and say why a citation is impossible without
  them.
- Measure **context recall** for a question set against a chunking strategy, rather than reading
  chunks and nodding.
- Find the question that no strategy answers, and diagnose whether it is a chunking problem or a
  property of the corpus.
- Sweep chunk size and explain why recall falls at both ends, for two different reasons.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تقطّع وثيقة بحجم ثابت، وبحدود الجمل، وبتغيّر الموضوع، وأن تقول ثمن كل قطع.
- أن تشرح ما يشتريه التداخل (Overlap) وما تدفعه ثمنًا له، برقم لا برأي.
- أن تُبقي المصدر والقسم ورقم المقطع على كل مقطع، وأن تقول لماذا يستحيل الاستشهاد بدونها.
- أن تقيس **استدعاء السياق (Context Recall)** لمجموعة أسئلة عند كل طريقة تقطيع، بدل قراءة المقاطع
  والإيماء برأسك.
- أن تجد السؤال الذي لا تجيب عنه أي طريقة، وأن تشخّص هل هو خلل تقطيع أم صفة في المُدوّنة نفسها.
- أن تمسح حجم المقطع وأن تشرح لماذا يهبط الاستدعاء عند الطرفين، ولسببين مختلفين.

</div>

## About the data

**`policy_docs`** — 24 short policy documents for **Olo Retail**, a company that does not exist.
Written for this course, CC0, about 4,900 words in total. Each document is markdown with numbered
`## ` section headings, and `manifest.csv` lists the id, title, category and last-updated date.

**Why a fictional company.** Nothing in these documents is in any model's training data. On
Wednesday that is what makes the hallucination demonstration honest — an ungrounded model *cannot*
get these answers right by accident. Today it means retrieval has nowhere to hide either: if the
answer comes out, it came out of a chunk you can point at.

**`rag_eval_questions`** — 40 questions over that corpus. Thirty are answerable and name the exact
sections that answer them, in the form `POL-114 §2`. Ten are deliberately out of scope and name
nothing: they exist so that Thursday can score a refusal. Today you only use the 30.

**The known problem with this data — two, in fact.**

1. **Two documents contradict each other.** POL-103 §3 says store credit expires after 12 months;
   POL-119 §2 says 24. Wednesday's last task is that disagreement. Today it means one question has
   two correct sources in two different documents.
2. **Three questions need two documents at once**, and the two documents share almost no
   vocabulary. Core task 6 is about one of them, and it is the most instructive row in the table.

**First-run download:** `sentence-transformers/all-MiniLM-L6-v2`, about 90 MB, cached after the
first time. You have used it in weeks 5 and 6.

<div dir="rtl" align="right">

## عن البيانات

**`policy_docs`** — أربع وعشرون وثيقة سياسات قصيرة لشركة **Olo Retail** التي لا وجود لها. كُتبت لهذا
المقرّر برخصة CC0، ومجموعها نحو ٤٩٠٠ كلمة. وكل وثيقة ملف markdown بعناوين أقسام مرقّمة تبدأ بـ`## `،
ويسرد `manifest.csv` المعرّف والعنوان والتصنيف وتاريخ آخر تحديث.

**لماذا شركة متخيَّلة؟** لأن لا شيء في هذه الوثائق موجود في بيانات تدريب أي نموذج. وهذا ما يجعل
عرض الهلوسة يوم الأربعاء أمينًا — إذ **لا يمكن** لنموذج غير مُسنَد أن يصيب هذه الإجابات مصادفةً.
ومعناه اليوم أن الاسترجاع بلا مخبأ أيضًا: فإن خرجت الإجابة فقد خرجت من مقطع تستطيع أن تشير إليه.

**`rag_eval_questions`** — أربعون سؤالًا على تلك المُدوّنة. ثلاثون قابلة للإجابة وتُسمّي الأقسام التي
تجيب عنها بالضبط، بالصيغة `POL-114 §2`. وعشرة خارج النطاق عمدًا ولا تُسمّي شيئًا، وهي موجودة ليقيس
يوم الخميس الرفض. واليوم لا تستعمل غير الثلاثين.

**والمشكلة المعروفة في هذه البيانات — بل مشكلتان:**

١. **وثيقتان تتناقضان.** فـ`POL-103 §3` تقول إن الرصيد المتجري ينتهي بعد ١٢ شهرًا، و`POL-119 §2`
   تقول ٢٤ شهرًا. وتلك المخالفة هي مهمة الأربعاء الأخيرة. ومعناها اليوم أن لسؤال واحد مصدرين
   صحيحين في وثيقتين مختلفتين.
٢. **وثلاثة أسئلة تحتاج وثيقتين معًا**، والوثيقتان لا تكادان تشتركان في مفردة. والمهمة السادسة عن
   واحد منها، وهو أكثر صفوف الجدول تعليمًا.

**تنزيل عند أول تشغيل:** النموذج `sentence-transformers/all-MiniLM-L6-v2` بنحو ٩٠ ميغابايت، ويُخزَّن
بعد أول مرة. وقد استعملته في الأسبوعين الخامس والسادس.

</div>

## Setup

One thing to notice in the setup cell: `SIZE`, `OVERLAP` and `MAX_CHARS` are named constants at the
top, not numbers buried in a function. Every chunking parameter in a real system ends up in a config
file, and the reason is the sweep you run in section 3 — you will change these numbers ten times
today.

<div dir="rtl" align="right">

## الإعداد

انتبه في خلية الإعداد إلى أن `SIZE` و`OVERLAP` و`MAX_CHARS` ثوابت مسمّاة في الأعلى لا أرقامًا مدفونة
داخل دالة. فكل معامل تقطيع في نظام حقيقي ينتهي إلى ملف إعدادات، والسبب هو المسح الذي تُجريه في القسم
الثالث — إذ ستغيّر هذه الأرقام عشر مرات اليوم.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, get_dataset_dir
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report
from aiep.viz import use_course_style, savefig

ensure("sentence-transformers", "scikit-learn", "matplotlib", "pandas", "pyarrow")
seed_everything(42)

import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

use_course_style()
np.set_printoptions(precision=3, suppress=True)

EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"
SIZE, OVERLAP = 400, 40        # characters, and 10% of them — section 2 changes both
MAX_CHARS = 600                # no chunk may exceed this; the sanity check enforces it
MIN_CHARS = 150                # sentence and semantic chunks merge up to at least this
TOP_K = 3                      # retrieve three chunks per question, all day

CORPUS_DIR = get_dataset_dir("policy_docs")
MANIFEST = pd.read_csv(CORPUS_DIR / "manifest.csv")
QUESTIONS = pd.read_parquet(get_dataset("rag_eval_questions"))
ANSWERABLE = QUESTIONS[QUESTIONS.answerable].reset_index(drop=True)

print(f"{len(MANIFEST)} documents, {MANIFEST.n_words.sum():,} words, "
      f"{MANIFEST.n_sections.sum()} sections")
print(f"{len(QUESTIONS)} questions — {len(ANSWERABLE)} answerable, "
      f"{len(QUESTIONS) - len(ANSWERABLE)} deliberately out of scope")
print(versions(), "| device:", device())

## Section 1 — Warm-up: the slide's passage, and its numbers  (≈25 min)

Everything in this section works. It is this morning's 120-word passage and the four measurements
the slides quoted, so that when the same code runs on 24 documents you already know what it does.

The retriever here is **TF-IDF**, the one you built in W5D1, on purpose. Retrieval is not the new
idea today; chunking is, and putting a second unknown next to it would blur which one moved.

Run the cell and check four things against your notes from this morning:

1. Fixed-size, 40 words, no overlap → **3 chunks**, and chunk 1 ends `within 14`.
2. Query 1 retrieves **chunk 2** at `0.069` against `0.034` — correctly, and uselessly.
3. Ten words of overlap → **4 chunks**, chunk 2 wins at `0.102`, and it holds the whole sentence.
4. Sentence chunking → **7 chunks**, and chunk 3 wins outright at `0.155`.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: فقرة الشريحة وأرقامها (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل. إنه فقرة المئة وعشرين كلمة من هذا الصباح والقياسات الأربعة التي ذكرتها
الشرائح، حتى إذا عمل الرمز نفسه على أربع وعشرين وثيقة كنت تعرف مسبقًا ماذا يفعل.

والمُسترجِع هنا **TF-IDF** الذي بنيته في الأسبوع الخامس اليوم الأول، عن قصد. فالاسترجاع ليس الفكرة
الجديدة اليوم بل التقطيع، ووضع مجهول ثانٍ بجواره يُلبِّس أيّهما تحرّك.

شغّل الخلية وطابق أربعة أشياء مع ملاحظاتك:

١. حجم ثابت بأربعين كلمة بلا تداخل ← **ثلاثة مقاطع**، وينتهي الأول بـ`within 14`.
٢. السؤال الأول يسترجع **المقطع الثاني** عند `0.069` مقابل `0.034` — استرجاعًا صحيحًا وعديم الفائدة.
٣. تداخل بعشر كلمات ← **أربعة مقاطع**، ويفوز الثاني عند `0.102` وفيه الجملة كاملة.
٤. التقطيع بالجمل ← **سبعة مقاطع**، ويفوز الثالث وحده عند `0.155`.

</div>

In [ ]:
# The passage from slide 38, character for character. It is also section 2 of POL-101 in
# the corpus you load below — the same words, so the warm-up and the lab measure one thing.
PASSAGE = (
    "Olo Retail accepts returns on most items sold through its online store and its branches "
    "in Riyadh, Jeddah and Dammam. Customers must bring the original receipt and the packaging. "
    "A refund is issued to the original payment method within 14 days of purchase. "
    "Digital goods are excluded from that window and cannot be refunded once the licence key "
    "has been revealed. Items that have been opened may still be exchanged at the discretion of "
    "the branch manager, who may also offer store credit instead of a refund. Exchanges follow "
    "the same schedule as refunds and are handled at the branch where the purchase was made. "
    "Gift purchases may be returned by the recipient with a gift receipt within that period.")

WORDS = PASSAGE.split()
Q1 = "how many days do I have to return an item"
Q2 = "how long do I have to return digital goods"


def chunk_words(text, size, overlap=0):
    """Fixed-size chunks counted in WORDS. Section 2 switches to characters; this is the slide."""
    words, out, i, step = text.split(), [], 0, size - overlap
    while i < len(words):
        out.append(" ".join(words[i:i + size]))
        if i + size >= len(words):
            break
        i += step
    return out


def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]


def tfidf_similarities(chunks, query):
    """W5D1's retriever: cosine against a TF-IDF space fitted on these chunks plus the query."""
    space = TfidfVectorizer().fit(chunks + [query])
    matrix = space.transform(chunks).toarray()
    vector = space.transform([query]).toarray()[0]
    norms = np.linalg.norm(matrix, axis=1) * np.linalg.norm(vector)
    return matrix @ vector / (norms + 1e-12)


no_overlap = chunk_words(PASSAGE, 40)
with_overlap = chunk_words(PASSAGE, 40, 10)
sentence_chunks = split_sentences(PASSAGE)

print(f"the passage is {len(WORDS)} words\n")
print(f"1. fixed 40, no overlap  → {len(no_overlap)} chunks")
print(f"   chunk 1 ends: ...{no_overlap[0][-24:]!r}")
print(f"   chunk 2 opens: {no_overlap[1][:24]!r}")

sims1 = tfidf_similarities(no_overlap, Q1)
best = int(sims1.argmax())
print(f"\n2. query 1 scores {sims1.round(3).tolist()} → chunk {best + 1} wins")
print(f"   'within 14 days' in the retrieved chunk: {'within 14 days' in no_overlap[best]}")
print("   the retriever was right, and the answer is in the other chunk")

sims1_overlap = tfidf_similarities(with_overlap, Q1)
best_overlap = int(sims1_overlap.argmax())
print(f"\n3. fixed 40, overlap 10  → {len(with_overlap)} chunks, "
      f"chunk {best_overlap + 1} wins at {sims1_overlap.max():.3f}")
print(f"   'within 14 days of purchase' in it: "
      f"{'within 14 days of purchase' in with_overlap[best_overlap]}")
print(f"   stored words: {len(WORDS)} → {sum(len(c.split()) for c in with_overlap)}")

sims1_sentence = tfidf_similarities(sentence_chunks, Q1)
best_sentence = int(sims1_sentence.argmax())
print(f"\n4. sentence chunking     → {len(sentence_chunks)} chunks, lengths "
      f"{[len(c.split()) for c in sentence_chunks]}")
print(f"   chunk {best_sentence + 1} wins at {sims1_sentence.max():.3f}, "
      f"{len(sentence_chunks[best_sentence].split())} words, all of them relevant")

### The reversal, and where a semantic chunker would cut

Sentence chunking looked like the winner on query 1. Now ask query 2 — *how long do I have to
return digital goods* — which needs **two facts from two sentences**: the fourteen-day window and
the exception that excludes digital goods from it.

The second cell measures the similarity between each pair of consecutive sentences. The weakest
link in the whole passage is **0.018**, and it sits exactly between those two sentences. A semantic
chunker would cut there, correctly — and query 2 has to read across that cut.

**That is the whole lesson of the morning in one number.** The best boundary by topic is the worst
boundary for a question that spans the topic.

<div dir="rtl" align="right">

### الانقلاب، والموضع الذي يقطع فيه المقطِّع الدلالي

بدا التقطيع بالجمل فائزًا في السؤال الأول. فاسأل الآن السؤال الثاني — *كم مدّة إرجاع السلع الرقمية*
— وهو يحتاج **حقيقتين من جملتين**: نافذة الأربعة عشر يومًا، والاستثناء الذي يُخرج السلع الرقمية منها.

وتقيس الخلية الثانية التشابه بين كل جملتين متتاليتين. وأضعف رابط في الفقرة كلها هو **0.018**، وهو
واقع بين تينك الجملتين تمامًا. فالمقطِّع الدلالي سيقطع هناك، وقطعه صحيح — وعلى السؤال الثاني أن
يقرأ عبر ذلك القطع.

**وهذا درس الصباح كله في رقم واحد:** أفضل حدٍّ موضوعيًّا هو أسوأ حدٍّ لسؤال يمتدّ عبر الموضوع.

</div>

In [ ]:
FACT_WINDOW = "within 14 days of purchase"
FACT_EXCEPTION = "Digital goods are excluded"

print("query 2 — which strategy retrieved BOTH facts?\n")
for name, chunks in [("fixed 40, no overlap", no_overlap),
                     ("fixed 40, overlap 10", with_overlap),
                     ("sentence", sentence_chunks)]:
    winner = chunks[int(tfidf_similarities(chunks, Q2).argmax())]
    has = (FACT_WINDOW in winner, FACT_EXCEPTION in winner)
    print(f"  {name:<22} window: {str(has[0]):<5} exception: {str(has[1]):<5} "
          f"→ {'answerable' if all(has) else 'NOT answerable'}")

# Where a semantic chunker would cut: the lowest similarity between consecutive sentences.
# The slide's curve is TF-IDF, the same retriever as above, so this reproduces its 0.018.
space = TfidfVectorizer().fit(sentence_chunks)
tfidf_matrix = space.transform(sentence_chunks).toarray()
adjacent_tfidf = np.array([a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)
                           for a, b in zip(tfidf_matrix, tfidf_matrix[1:])])
weakest = int(adjacent_tfidf.argmin())

print(f"\nadjacent-sentence similarity, TF-IDF: {adjacent_tfidf.round(3).tolist()}")
print(f"weakest link {adjacent_tfidf.min():.3f}, between sentence {weakest + 1} and {weakest + 2}:")
print(textwrap.fill(f"  … {sentence_chunks[weakest]}", 96))
print(textwrap.fill(f"  … {sentence_chunks[weakest + 1]}", 96))
print("\nA semantic chunker cuts at the lowest point on that curve — right where query 2 reads.")

# Now the same curve with sentence embeddings, which is what task 2.3 will actually use.
model = SentenceTransformer(EMBEDDER)
sentence_vectors = model.encode(sentence_chunks, normalize_embeddings=True)
adjacent_embed = (sentence_vectors[:-1] * sentence_vectors[1:]).sum(axis=1)
print(f"\nthe same curve with embeddings:      {adjacent_embed.round(3).tolist()}")
print(f"its weakest link is {adjacent_embed.min():.3f}, between sentence "
      f"{int(adjacent_embed.argmin()) + 1} and {int(adjacent_embed.argmin()) + 2} — "
      f"a different boundary.")
print("Two signals, one passage, two answers about where the topic changed. The chunker is not\n"
      "finding a fact about the text; it is reporting what its own representation notices.")

WARMUP = {"n_nooverlap": len(no_overlap), "n_overlap": len(with_overlap),
          "n_sentence": len(sentence_chunks), "q1_nooverlap": float(sims1.max()),
          "q1_overlap": float(sims1_overlap.max()), "q1_sentence": float(sims1_sentence.max()),
          "weakest_link": float(adjacent_tfidf.min()),
          "weakest_link_embed": float(adjacent_embed.min()),
          "chunk1_ends_within_14": no_overlap[0].endswith("within 14")}

## Section 2 — Core: six tasks  (≈60 min)

1. Overlap, on the passage: confirm the sentence survives, and count what it cost.
2. `chunk_sentences` on a real document, with a minimum size so single-clause chunks do not pollute
   the index.
3. `chunk_semantic` — cut where consecutive sentences stop being similar.
4. Chunk all 24 documents four ways, **keeping metadata on every chunk**.
5. **The measurement.** Context recall at k=3, for 30 questions × 4 strategies.
6. Read the table: one question every strategy answers, one that only some do, and one that
   **none** fully answers.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. التداخل على الفقرة: تأكّد أن الجملة نجت، واحسب ثمنه.
٢. `chunk_sentences` على وثيقة حقيقية، بحدٍّ أدنى للحجم كي لا تلوّث المقاطع أحادية العبارة الفهرس.
٣. `chunk_semantic` — اقطع حيث يتوقّف تشابه الجمل المتتالية.
٤. قطّع الوثائق الأربع والعشرين بأربع طرق، **مع إبقاء البيانات الوصفية على كل مقطع**.
٥. **القياس.** استدعاء السياق عند ٣، لثلاثين سؤالًا × أربع طرق.
٦. اقرأ الجدول: سؤال تجيب عنه كل طريقة، وسؤال تجيب عنه بعضها، وسؤال **لا تجيب عنه أيّ** منها كاملًا.

</div>

### Task 2.1 — overlap, and its bill

Re-cut the passage at 40 words with 10 words of overlap and confirm two things: the return-window
sentence now survives intact inside one chunk, and the corpus got bigger. Print both numbers.

The second number is the point. Overlap is free to type and not free to run: more chunks to embed,
more text stored, and duplicate hits in the same result set.

<div dir="rtl" align="right">

### المهمة ٢٫١ — التداخل وفاتورته

أعد قطع الفقرة بأربعين كلمة وعشر كلمات تداخل، وتأكّد من أمرين: أن جملة نافذة الإرجاع صارت كاملة
داخل مقطع واحد، وأن المُدوّنة كبرت. اطبع الرقمين.

والرقم الثاني هو المقصود. فالتداخل مجاني في الكتابة لا في التشغيل: مقاطع أكثر تُضمَّن، ونصّ أكثر
يُخزَّن، ونتائج مكرّرة في المجموعة الواحدة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) You already have chunk_words(text, size, overlap) from the warm-up — call it with
#    an overlap of 10 and check whether the whole sentence "within 14 days of purchase"
#    now sits inside one chunk.
# 2) The cost is in two numbers: how many chunks you now have, and how many words are
#    stored in total across them. Compare both against the no-overlap split.
# 3) Say the cost out loud in a print: overlap is stored twice and embedded twice.
# Search: "text chunking overlap sliding window retrieval"
# https://www.sbert.net/examples/applications/semantic-search/README.html
#
# ١) لديك `chunk_words(text, size, overlap)` من الإحماء — نادِها بتداخل ١٠ وتحقّق هل صارت
#    الجملة الكاملة "within 14 days of purchase" داخل مقطع واحد.
# ٢) الثمن في رقمين: كم مقطعًا صار عندك، وكم كلمة تُخزَّن فيها جميعًا. قارن الرقمين
#    بالقطع بلا تداخل.
# ٣) قل الثمن صراحةً في `print`: التداخل يُخزَّن مرّتين
#    ويُضمَّن مرّتين.
# ابحث عن: "text chunking overlap sliding window retrieval"
# https://www.sbert.net/examples/applications/semantic-search/README.html
# ────────────────────────────────────────────────────────────────────

print(f"the full sentence survives in {len(SURVIVES)} of {len(with_overlap)} chunks")
print(f"chunks:  {len(no_overlap)} → {len(with_overlap)}")
print(f"words stored: {words_plain} → {words_overlap} "
      f"(+{100 * (words_overlap - words_plain) / words_plain:.0f}%)")
print("you pay that percentage again at every embedding, and again in storage")

### Task 2.2 — sentence chunks, with a floor

Sentence boundaries are already in the text; use them. But a naive sentence split produces chunks
like *"Points appear on the account within 24 hours."* — nine words, no context, and it will match
every question about points weakly.

So merge consecutive sentences until the chunk reaches at least `MIN_CHARS`. Write
`chunk_sentences(text, min_chars)`, run it on one document, and print the chunk-length distribution
for it and for the fixed-size split beside it.

**Look at the two distributions before you move on.** Fixed-size gives you a spike at the size you
chose; sentence chunking gives you a spread. Neither is better in the abstract, and the table in
task 5 is the only thing that decides.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — مقاطع الجمل، بحدٍّ أدنى

حدود الجمل موجودة في النصّ أصلًا، فاستعملها. لكن التقسيم الساذج بالجمل يُنتج مقاطع مثل *«تظهر
النقاط في الحساب خلال ٢٤ ساعة»* — تسع كلمات بلا سياق، وستطابق كل سؤال عن النقاط مطابقةً ضعيفة.

فادمج الجمل المتتالية حتى يبلغ المقطع `MIN_CHARS` على الأقل. اكتب `chunk_sentences(text, min_chars)`،
وشغّلها على وثيقة واحدة، واطبع توزيع أطوال المقاطع لها وللقطع ثابت الحجم بجوارها.

**وانظر إلى التوزيعين قبل أن تمضي.** فالحجم الثابت يعطيك قمّة عند الحجم الذي اخترته، والتقطيع بالجمل
يعطيك انتشارًا. ولا أفضلية لأحدهما في المطلق، ولا يحكم بينهما إلا جدول المهمة الخامسة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Read a document with read_document(doc_id) below — it returns the full text and a
#    list of (start, end, section_number) spans, so a chunk can say which section it is in.
# 2) Split into sentences, then accumulate them into a buffer and emit the buffer only
#    once it is at least min_chars long. Emit whatever is left at the end.
# 3) Return (start, end) character spans rather than strings: task 2.4 needs the span to
#    work out which sections a chunk covers, and the text is text[start:end] anyway.
# Search: "python split text into sentences regex lookbehind"
# https://docs.python.org/3/library/re.html
#
# ١) اقرأ الوثيقة بـ`read_document(doc_id)` أدناه — تُعيد النصّ الكامل وقائمة مجالات
#    `(start, end, section)` كي يعرف المقطع في أي قسم هو.
# ٢) قسّم إلى جمل، ثم اجمعها في مخزن مؤقّت ولا تُخرجه إلا إذا بلغ `min_chars`. وأخرج ما
#    تبقّى في النهاية.
# ٣) أعِد مجالات المحارف `(start, end)` لا النصوص: فالمهمة ٢٫٤ تحتاج المجال لتعرف الأقسام
#    التي يغطّيها المقطع، والنصّ هو `text[start:end]` أصلًا.
# ابحث عن: "python split text into sentences regex lookbehind"
# https://docs.python.org/3/library/re.html
# ────────────────────────────────────────────────────────────────────

def read_document(doc_id):
    """Full text of one document, plus (start, end, section_number) for each section."""
    lines = (CORPUS_DIR / "docs" / f"{doc_id}.md").read_text(encoding="utf-8").splitlines()
    sections, current = [], None
    for line in lines:
        if line.startswith("## "):
            current = [line[3:].strip().split(". ", 1)[-1], []]
            sections.append(current)
        elif line.startswith("#") or line.startswith("*") or not line.strip():
            continue
        elif current is not None:
            current[1].append(line.strip())
    text, spans, at = "", [], 0
    for number, (heading, body) in enumerate(sections, start=1):
        part = f"{heading}. {' '.join(body)} "
        spans.append((at, at + len(part), number))
        text += part
        at += len(part)
    return text, spans
def chunk_fixed(text, size, overlap=0):
    """Fixed-size chunks counted in characters, returned as (start, end) spans."""
    spans, at, step = [], 0, size - overlap
    while at < len(text):
        spans.append((at, min(at + size, len(text))))
        if at + size >= len(text):
            break
        at += step
    return spans
# TODO: Write chunk_sentences(text, min_chars): split on sentence boundaries, then merge consecutive sentences until the chunk is at least min_chars long. Return character spans.
# مهمة: اكتب `chunk_sentences(text, min_chars)`: قسّم عند حدود الجمل ثم ادمج الجمل المتتالية حتى يبلغ المقطع `min_chars`. وأعِد مجالات المحارف.

### Task 2.3 — semantic chunks: cut where the topic changes

Embed each sentence, measure the similarity between consecutive sentences, and start a new chunk
where that similarity drops below a threshold. Keep the same `MIN_CHARS` floor, or one stray short
sentence produces one stray short chunk.

Then **look at two boundaries by eye** and say whether you agree with them. This is the only task
today where the honest answer is a judgement rather than a number, and it is worth five minutes:
a semantic chunker is a model making an editorial decision about your documents, at ingestion,
once, for everything you will ever ask.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — المقاطع الدلالية: اقطع حيث يتغيّر الموضوع

ضمّن كل جملة، وقِس التشابه بين الجمل المتتالية، وابدأ مقطعًا جديدًا حيث يهبط هذا التشابه دون عتبة.
وأبقِ الحدّ الأدنى `MIN_CHARS` نفسه، وإلا أنتجت جملةٌ قصيرةٌ شاردة مقطعًا قصيرًا شاردًا.

ثم **انظر إلى حدَّين بعينك** وقل هل توافق عليهما. وهذه المهمة الوحيدة اليوم التي جوابها الأمين حُكم لا
رقم، وتستحقّ خمس دقائق: فالمقطِّع الدلالي نموذجٌ يتّخذ قرارًا تحريريًّا في وثائقك، عند الإدخال، مرّةً
واحدة، ولكل ما ستسأل عنه أبدًا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Split into sentences and keep each one's (start, end) span, the same way task 2.2 did.
# 2) Encode the sentences with model.encode(..., normalize_embeddings=True); the cosine
#    between neighbours is then just the row-wise product of consecutive vectors, summed.
# 3) Walk the sentences, extending the current chunk, and close it when the similarity to
#    the next sentence is below THRESHOLD *and* the chunk already meets min_chars.
# Search: "semantic chunking embedding similarity between consecutive sentences"
# https://www.sbert.net/docs/package_reference/SentenceTransformer.html
# Note: the embedding threshold is not the slide's TF-IDF threshold — expect the
# boundaries to land in different places.
#
# ١) قسّم إلى جمل واحتفظ بمجال `(start, end)` لكل جملة، كما في المهمة ٢٫٢.
# ٢) ضمّن الجمل بـ`model.encode(..., normalize_embeddings=True)`، فيصير جيب التمام بين
#    المتجاورتين حاصلَ ضربٍ عنصريًّا للمتّجهين المتتاليين مجموعًا.
# ٣) امشِ على الجمل مُوسِّعًا المقطع الحالي، وأغلقه حين يهبط التشابه مع الجملة التالية دون
#    `THRESHOLD` **و**يكون المقطع قد بلغ `min_chars`.
# ابحث عن: "semantic chunking embedding similarity between consecutive sentences"
# https://www.sbert.net/docs/package_reference/SentenceTransformer.html
# ملاحظة: عتبة التضمينات ليست عتبة TF-IDF التي في الشريحة، فتوقّع أن تقع
# الحدود في مواضع مختلفة.
# ────────────────────────────────────────────────────────────────────

THRESHOLD = 0.35        # below this, consecutive sentences are about different things
# TODO: Write chunk_semantic(text, threshold, min_chars): embed the sentences, cut where the similarity between neighbours drops below threshold, and keep the min_chars floor.
# مهمة: اكتب `chunk_semantic(text, threshold, min_chars)`: ضمّن الجمل، واقطع حيث يهبط التشابه بين المتجاورتين دون العتبة، مع إبقاء الحدّ الأدنى `min_chars`.
for a, b in demo_semantic[:2]:
    print(textwrap.fill(f"› {demo_text[a:b]}", 96), "\n")
print("Do you agree with where it cut? Write one sentence in the markdown cell below.")

**Your answer — two boundaries, in one sentence each.** Replace this text.

> Boundary 1: …
>
> Boundary 2: …

<div dir="rtl" align="right">

**جوابك — حدّان، جملة لكل حدّ.** استبدل هذا النص.

> الحدّ الأول: …
>
> الحدّ الثاني: …

</div>

### Task 2.4 — chunk the whole corpus, four ways, with metadata

Now run all four strategies over all 24 documents and build one table with a row per chunk:
`strategy`, `doc_id`, `chunk_index`, `sections`, `start`, `end`, `text`.

**`sections` is the one that matters.** A chunk that spans the end of section 2 and the start of
section 3 covers both, and a citation has to say so. This is the metadata Wednesday's prompt
demands and Thursday's evaluation scores against — throw it away here and the only fix is to
re-ingest the entire corpus.

Assert two things before you go on: every chunk has a non-empty `sections`, and no chunk exceeds
`MAX_CHARS`. Both failures are silent otherwise, and both are the kind you find three days later.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — قطّع المُدوّنة كلها بأربع طرق مع البيانات الوصفية

شغّل الطرق الأربع على الوثائق الأربع والعشرين جميعًا، وابنِ جدولًا واحدًا فيه صفّ لكل مقطع:
`strategy` و`doc_id` و`chunk_index` و`sections` و`start` و`end` و`text`.

**و`sections` هو المهمّ.** فالمقطع الذي يمتدّ من آخر القسم الثاني إلى أول الثالث يغطّيهما معًا، وعلى
الاستشهاد أن يقول ذلك. وهذه هي البيانات الوصفية التي يطلبها موجّه الأربعاء ويقيس عليها تقييم
الخميس — فإن أهملتها هنا فلا علاج إلا بإعادة إدخال المُدوّنة كلها.

وأكّد أمرين قبل أن تمضي: أن لكل مقطع `sections` غير فارغ، وأن لا مقطع يتجاوز `MAX_CHARS`. وإلا كان
الإخفاقان صامتين، وكلاهما من النوع الذي تكتشفه بعد ثلاثة أيام.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Write one build_chunks(name, splitter) that loops over MANIFEST.doc_id, calls
#    read_document, applies the splitter to get spans, and emits one row per span.
# 2) A chunk covers a section when the two character ranges overlap at all: the chunk
#    starts before the section ends AND the section starts before the chunk ends.
# 3) Store sections as a sorted, "|"-joined string like "POL-114 §2|POL-114 §3" so the
#    column survives a parquet round-trip and D2 can read it back without a parser.
# Search: "python interval overlap test start end"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html
#
# ١) اكتب دالة واحدة `build_chunks(name, splitter)` تدور على `MANIFEST.doc_id`، وتنادي
#    `read_document`، وتطبّق المُقطِّع للحصول على المجالات، وتُخرج صفًّا لكل مجال.
# ٢) يغطّي المقطعُ القسمَ إذا تقاطع المجالان أصلًا: يبدأ المقطع قبل نهاية القسم **و**يبدأ
#    القسم قبل نهاية المقطع.
# ٣) خزّن الأقسام سلسلةً مرتّبة موصولة بـ"|" مثل `"POL-114 §2|POL-114 §3"` كي يبقى العمود
#    بعد الحفظ في parquet ويقرأه اليوم الثاني بلا مُحلِّل.
# ابحث عن: "python interval overlap test start end"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html
# ────────────────────────────────────────────────────────────────────

STRATEGIES = {
    "fixed_nooverlap": lambda text: chunk_fixed(text, SIZE, 0),
    "fixed_overlap": lambda text: chunk_fixed(text, SIZE, OVERLAP),
    "sentence": chunk_sentences,
    "semantic": chunk_semantic,
}
# TODO: Write covered_sections(doc_id, spans, start, end) returning the set of section ids a chunk touches, formatted as "POL-114 §2".
# مهمة: اكتب `covered_sections(doc_id, spans, start, end)` تُعيد مجموعة معرّفات الأقسام التي يلمسها المقطع، بالصيغة `"POL-114 §2"`.
# TODO: Build one DataFrame of every chunk under every strategy, with the metadata columns.
# مهمة: ابنِ إطار بيانات واحدًا لكل مقطع عند كل طريقة، مع أعمدة البيانات الوصفية.
CHUNKS["n_chars"] = CHUNKS.text.str.len()
summary = CHUNKS.groupby("strategy").agg(chunks=("text", "size"),
                                         mean_chars=("n_chars", "mean"),
                                         max_chars=("n_chars", "max"),
                                         spanning=("sections",
                                                   lambda s: s.str.contains(r"\|").sum()))
print(summary.round(0).to_string())
print(f"\n'spanning' counts chunks that cover more than one section — those are the ones a "
      f"citation gets wrong if you store only one source id.")
assert (CHUNKS.sections.str.len() > 0).all(), "a chunk lost its section metadata"
assert CHUNKS.n_chars.max() <= MAX_CHARS, f"a chunk is {CHUNKS.n_chars.max()} chars"

### Task 2.5 — the measurement

This is the task. Everything before it was setup.

For each of the 30 answerable questions and each of the four strategies: embed the question, embed
the chunks, take the **top 3** by cosine, collect every section those three chunks cover, and score

    context recall = |retrieved sections ∩ gold sections| / |gold sections|

against the question's `gold_sections`. Note that this is scored **at section level, not document
level**. Retrieving the right document and the wrong section is the most common RAG failure there
is, and a document-level metric scores it as a success.

Build `chunk_comparison.parquet`: one row per question per strategy — `strategy`, `qid`, `kind`,
`recall_at_3`, `retrieved_sections`, `retrieved_chunk_ids`. That is 30 × 4 = 120 rows, plus the 10
out-of-scope questions carried with a null recall so Thursday can find them: 160 rows in all.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — القياس

هذه هي المهمّة. وكل ما قبلها تهيئة.

لكل سؤال من الثلاثين القابلة للإجابة، ولكل طريقة من الأربع: ضمّن السؤال، وضمّن المقاطع، وخذ **أفضل
ثلاثة** بجيب التمام، واجمع كل الأقسام التي تغطّيها تلك الثلاثة، ثم احسب

    استدعاء السياق = |الأقسام المسترجَعة ∩ الأقسام المرجعية| ÷ |الأقسام المرجعية|

مقابل `gold_sections` للسؤال. ولاحظ أن القياس **على مستوى القسم لا الوثيقة**. فاسترجاع الوثيقة
الصحيحة والقسم الخطأ أشيع إخفاقات RAG، والمقياس على مستوى الوثيقة يعدّه نجاحًا.

وابنِ `chunk_comparison.parquet`: صفّ لكل سؤال عند كل طريقة — `strategy` و`qid` و`kind`
و`recall_at_3` و`retrieved_sections` و`retrieved_chunk_ids`. أي ٣٠ × ٤ = ١٢٠ صفًّا، مع العشرة خارج
النطاق محمولةً باستدعاء فارغ ليجدها الخميس: ١٦٠ صفًّا جميعًا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Encode every question once (all 40), and each strategy's chunks once. Normalising
#    both means the cosine is a plain matrix product: questions @ chunks.T.
# 2) For one question and one strategy, np.argsort(-scores)[:TOP_K] gives the three chunk
#    rows; union their "sections" strings to get everything those chunks cover.
# 3) Recall is the size of the intersection with the gold set divided by the size of the
#    gold set. Leave it as None for the out-of-scope questions — they have no gold.
# Search: "sentence transformers encode normalize_embeddings cosine similarity matrix"
# https://www.sbert.net/examples/applications/semantic-search/README.html
#
# ١) ضمّن الأسئلة كلها مرّة واحدة (الأربعين)، ومقاطع كل طريقة مرّة واحدة. وتطبيع الاثنين
#    يجعل جيب التمام حاصل ضرب مصفوفتين: `questions @ chunks.T`.
# ٢) لسؤال واحد وطريقة واحدة، يعطيك `np.argsort(-scores)[:TOP_K]` صفوف المقاطع الثلاثة؛
#    ووحّد سلاسل `sections` فيها لتعرف ما تغطّيه.
# ٣) الاستدعاء هو حجم التقاطع مع المجموعة المرجعية مقسومًا على حجمها. واتركه `None`
#    للأسئلة خارج النطاق — فلا مرجع لها.
# ابحث عن: "sentence transformers encode normalize_embeddings cosine similarity matrix"
# https://www.sbert.net/examples/applications/semantic-search/README.html
# ────────────────────────────────────────────────────────────────────

QUESTION_VECTORS = model.encode(QUESTIONS.question.tolist(),
                                normalize_embeddings=True, batch_size=64)
# TODO: For every strategy, embed its chunks, score all 40 questions, take the top TOP_K, and compute context recall at section level against gold_sections.
# مهمة: لكل طريقة، ضمّن مقاطعها، واحسب درجات الأسئلة الأربعين، وخذ أفضل `TOP_K`، واحسب استدعاء السياق على مستوى القسم مقابل `gold_sections`.
MEAN_RECALL = (COMPARISON[COMPARISON.answerable]
               .groupby("strategy").recall_at_3.mean().sort_values(ascending=False))
print(f"{len(COMPARISON)} rows — mean context recall@{TOP_K} on the 30 answerable questions:\n")
print(MEAN_RECALL.round(3).to_string())
print(f"\noverlap moved the mean by "
      f"{MEAN_RECALL['fixed_overlap'] - MEAN_RECALL['fixed_nooverlap']:+.3f}, and one question "
      f"is worth {1 / len(ANSWERABLE):.3f}.")
print("Read that before you conclude anything: on 30 questions this experiment cannot resolve "
      "a difference smaller than one question. The warm-up proved overlap fixes a severed "
      "sentence; this table cannot prove it pays at corpus scale, and saying so is the finding.")

### Task 2.6 — read the table, do not summarise it

A mean is not a finding. Pivot the comparison so each row is a question and each column a strategy,
then find three specific rows and write **one sentence about each**:

1. A question **every** strategy answers perfectly. Why is it easy?
2. A question where the strategies **disagree**. What does the winner's chunk contain that the
   loser's does not?
3. A question that **no strategy answers fully**. This is the instructive one, and it is usually a
   question whose answer lives in two documents that share no vocabulary — top-3 chunks cannot
   reach both, and no chunk size fixes that. Say what would.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — اقرأ الجدول ولا تُلخّصه

المتوسّط ليس نتيجة. حوّل الجدول ليكون كل صفّ سؤالًا وكل عمود طريقة، ثم جِد ثلاثة صفوف بعينها واكتب
**جملة واحدة عن كل منها**:

١. سؤال تجيب عنه **كل** الطرق إجابةً تامّة. لماذا كان سهلًا؟
٢. سؤال **تختلف** فيه الطرق. ماذا في مقطع الفائز وليس في مقطع الخاسر؟
٣. سؤال **لا تجيب عنه أيّ طريقة كاملًا**. وهذا هو المُعلِّم، وغالبًا يكون سؤالًا إجابته في وثيقتين لا
   تشتركان في مفردة — فلا تبلغ المقاطع الثلاثة كلتيهما، ولا يُصلح ذلك أي حجم مقطع. فقل ما الذي
   يُصلحه.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) DataFrame.pivot(index="qid", columns="strategy", values="recall_at_3") on the
#    answerable rows gives you the table to read.
# 2) Rows where every column is 1.0 are the easy ones; rows where the max is below 1.0
#    are the ones nothing answers; rows in between are the disagreements.
# 3) For the hardest row, print the question, its gold sections, and what each strategy
#    actually retrieved. The diagnosis is in that comparison, not in the score.
# Search: "pandas pivot compare rows across columns"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html
#
# ١) `DataFrame.pivot(index="qid", columns="strategy", values="recall_at_3")` على الصفوف
#    القابلة للإجابة يعطيك الجدول الذي تقرؤه.
# ٢) الصفوف التي كل أعمدتها ١٫٠ هي السهلة؛ والتي أقصاها دون ١٫٠ لا تجيب عنها أي طريقة؛
#    وما بينهما هي مواضع الخلاف.
# ٣) وللصفّ الأصعب، اطبع السؤال وأقسامه المرجعية وما استرجعته كل طريقة فعلًا. فالتشخيص في
#    تلك المقارنة لا في الدرجة.
# ابحث عن: "pandas pivot compare rows across columns"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html
# ────────────────────────────────────────────────────────────────────

# TODO: Pivot the answerable rows into question × strategy, then pick out the perfect rows, the disagreements, and the rows no strategy answers fully.
# مهمة: حوّل الصفوف القابلة للإجابة إلى سؤال × طريقة، ثم استخرج الصفوف التامّة، ومواضع الخلاف، والصفوف التي لا تجيب عنها أي طريقة كاملًا.
print(f"{len(PERFECT)} questions every strategy answers · {len(DISAGREE)} disagreements · "
      f"{len(UNANSWERED)} nothing answers fully\n")
print(PIVOT.loc[DISAGREE + UNANSWERED].round(2).to_string())
hardest = UNANSWERED[0]
row = QUESTIONS[QUESTIONS.qid == hardest].iloc[0]
print(f"\n{hardest} ({row.kind}): {row.question}")
print(f"  gold:      {row.gold_sections}")
for _, got in COMPARISON[(COMPARISON.qid == hardest)].iterrows():
    print(f"  {got.strategy:<16} retrieved {got.retrieved_sections}")

**Your three sentences.** Replace this text.

> Every strategy answers … because …
>
> The strategies disagree on … because the winning chunk contains …
>
> Nothing answers … The cause is … and what would fix it is …

<div dir="rtl" align="right">

**جملك الثلاث.** استبدل هذا النص.

> تجيب كل الطرق عن … لأن …
>
> تختلف الطرق في … لأن المقطع الفائز يحوي …
>
> ولا تجيب أي طريقة عن … وسببه … والذي يُصلحه هو …

</div>

## Section 3 — Stretch: sweep the size  (≈30 min)

Chunk the corpus at **100, 250, 500 and 1000 characters** with 10% overlap each time, measure mean
context recall for each, and plot recall against size.

The curve peaks somewhere in the middle. Then answer the question in a markdown cell:

**Why does recall fall at both ends — and why are the two reasons different?**

Small chunks fail one way and large chunks fail another, and a student who can only name one of the
two will pick the wrong fix roughly half the time. The capstone link is direct: if your project
retrieves anything — documents, tickets, product descriptions — this sweep is the first experiment
you run, and the number you choose is a number you can now defend.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: امسح الحجم (نحو ٣٠ دقيقة)

قطّع المُدوّنة عند **١٠٠ و٢٥٠ و٥٠٠ و١٠٠٠ محرف** بتداخل ١٠٪ في كل مرّة، وقِس متوسّط استدعاء السياق
لكلٍّ، وارسم الاستدعاء مقابل الحجم.

يبلغ المنحنى ذروته في مكان ما في الوسط. ثم أجب في خلية markdown:

**لماذا يهبط الاستدعاء عند الطرفين — ولماذا السببان مختلفان؟**

فالمقاطع الصغيرة تُخفق بطريقة والكبيرة بأخرى، ومن لا يعرف إلا أحد السببين سيختار العلاج الخطأ في
نصف الحالات تقريبًا. وصلته بمشروع التخرّج مباشرة: إن كان مشروعك يسترجع شيئًا — وثائق أو تذاكر أو
أوصاف منتجات — فهذا المسح أول تجربة تُجريها، والرقم الذي تختاره صار رقمًا تستطيع الدفاع عنه.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Loop over the four sizes; for each, build the chunk table with chunk_fixed at that
#    size and an overlap of size // 10, and reuse the covered_sections logic.
# 2) Reuse measure(): it takes a chunk table and the question vectors and does the rest.
#    Give each sweep its own strategy name so the rows do not collide.
# 3) Plot mean recall against size and mark the peak. Print the number of chunks too —
#    that is the cost axis, and it moves in the opposite direction.
# Search: "matplotlib plot line marker annotate peak"
# https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html
#
# ١) دُر على الأحجام الأربعة؛ وابنِ لكلٍّ جدول المقاطع بـ`chunk_fixed` عند ذلك الحجم
#    وتداخلٍ مقداره `size // 10`، وأعِد استعمال منطق `covered_sections`.
# ٢) أعِد استعمال `measure()`: فهي تأخذ جدول المقاطع ومتّجهات الأسئلة وتتولّى الباقي.
#    وأعطِ كل مسح اسم طريقة خاصًّا كي لا تتصادم الصفوف.
# ٣) ارسم متوسّط الاستدعاء مقابل الحجم وعلّم الذروة. واطبع عدد المقاطع أيضًا — فهو محور
#    الكلفة، وهو يتحرّك في الاتجاه المعاكس.
# ابحث عن: "matplotlib plot line marker annotate peak"
# https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html
# ────────────────────────────────────────────────────────────────────

SWEEP_SIZES = [100, 250, 500, 1000]
# TODO: Chunk the corpus at each size with 10% overlap, measure mean recall, and collect the results into one frame of size, chunks, mean recall.
# مهمة: قطّع المُدوّنة عند كل حجم بتداخل ١٠٪، وقِس متوسّط الاستدعاء، واجمع النتائج في إطار واحد فيه الحجم وعدد المقاطع ومتوسّط الاستدعاء.
print(SWEEP.round(3).to_string(index=False))
peak = SWEEP.loc[SWEEP.mean_recall.idxmax()]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(SWEEP["size"], SWEEP.mean_recall, marker="o")
ax.axvline(peak["size"], linestyle="--", linewidth=1)
ax.set_xlabel("chunk size (characters)")
ax.set_ylabel(f"mean context recall@{TOP_K}")
ax.set_title(f"recall peaks at {int(peak['size'])} characters")
savefig(fig, "chunk_size_sweep.png")
plt.show()

**Why does recall fall at both ends?** Replace this text with two short paragraphs — one per end.

> Small chunks: …
>
> Large chunks: …

<div dir="rtl" align="right">

**لماذا يهبط الاستدعاء عند الطرفين؟** استبدل هذا النص بفقرتين قصيرتين، واحدة لكل طرف.

> المقاطع الصغيرة: …
>
> المقاطع الكبيرة: …

</div>

## Save the artefact

`chunk_comparison.parquet` is tomorrow's input: D2 indexes the strategy that won here, and it reads
the winner out of this file rather than being told. The chunk table itself goes with it, so D2 does
not have to re-chunk the corpus to build an index over it.

<div dir="rtl" align="right">

## احفظ المُخرَج

`chunk_comparison.parquet` هو مُدخَل الغد: فاليوم الثاني يفهرس الطريقة الفائزة هنا، ويقرأ الفائز من
هذا الملف لا من إخبارٍ به. ويذهب جدول المقاطع معه كي لا يضطرّ اليوم الثاني إلى إعادة تقطيع المُدوّنة
ليبني عليها فهرسًا.

</div>

In [ ]:
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON.to_parquet(ARTEFACT_DIR / "chunk_comparison.parquet", index=False)
CHUNKS.to_parquet(ARTEFACT_DIR / "chunks.parquet", index=False)

WINNER = MEAN_RECALL.idxmax()
print(f"saved {len(COMPARISON)} comparison rows and {len(CHUNKS)} chunks to {ARTEFACT_DIR}")
print(f"winning strategy: {WINNER} at {MEAN_RECALL.max():.3f} — D2 indexes this one")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(WARMUP["chunk1_ends_within_14"] and WARMUP["n_nooverlap"] == 3,
      f"the 40-word no-overlap split must give 3 chunks with chunk 1 ending 'within 14' — got "
      f"{WARMUP['n_nooverlap']} chunks, ends correctly: {WARMUP['chunk1_ends_within_14']}. That "
      f"severed sentence is the evidence the whole morning rests on",
      f"يجب أن يعطي القطع بأربعين كلمة بلا تداخل ٣ مقاطع ينتهي أولها بـ'within 14' — والناتج "
      f"{WARMUP['n_nooverlap']} مقاطع، والنهاية صحيحة: {WARMUP['chunk1_ends_within_14']}. وتلك "
      f"الجملة المشقوقة هي الدليل الذي يقوم عليه الصباح كله")

check_close(WARMUP["q1_overlap"], 0.102,
            "query 1 with 10 words of overlap must score 0.102 as on slide 46",
            "يجب أن يسجّل السؤال الأول بتداخل عشر كلمات 0.102 كما في الشريحة ٤٦",
            tol=0.005)

check_close(WARMUP["weakest_link"], 0.018,
            "the weakest adjacent-sentence link must be 0.018, between the return window and the "
            "digital-goods exception — that is where a semantic chunker cuts",
            "يجب أن يكون أضعف رابط بين جملتين متتاليتين 0.018، بين نافذة الإرجاع واستثناء السلع "
            "الرقمية — وهناك يقطع المقطِّع الدلالي",
            tol=0.005)

check(bool((CHUNKS.sections.str.len() > 0).all()) and int(CHUNKS.n_chars.max()) <= MAX_CHARS,
      f"every chunk must carry its source and section metadata and stay under {MAX_CHARS} "
      f"characters — the longest is {int(CHUNKS.n_chars.max())}. Without the metadata, "
      f"Wednesday's citations cannot be checked by anyone",
      f"يجب أن يحمل كل مقطع مصدره وقسمه وأن يبقى دون {MAX_CHARS} محرفًا — وأطولها "
      f"{int(CHUNKS.n_chars.max())}. وبلا البيانات الوصفية لا يستطيع أحد التحقّق من استشهادات الأربعاء")

check(len(COMPARISON) == len(QUESTIONS) * len(STRATEGIES),
      f"chunk_comparison must hold every question against every strategy — expected "
      f"{len(QUESTIONS) * len(STRATEGIES)} rows, got {len(COMPARISON)}",
      f"يجب أن يحوي `chunk_comparison` كل سؤال عند كل طريقة — والمتوقّع "
      f"{len(QUESTIONS) * len(STRATEGIES)} صفًّا، والناتج {len(COMPARISON)}")

OVERLAP_DELTA = MEAN_RECALL["fixed_overlap"] - MEAN_RECALL["fixed_nooverlap"]
check(abs(OVERLAP_DELTA) <= 1.5 / len(ANSWERABLE),
      f"overlap moved mean recall by {OVERLAP_DELTA:+.3f}, which is more than one question's "
      f"worth ({1 / len(ANSWERABLE):.3f}) on {len(ANSWERABLE)} questions. On this corpus the two "
      f"fixed splits are supposed to be indistinguishable at this size — a large move means the "
      f"chunking changed something other than the boundaries",
      f"حرّك التداخل متوسّط الاستدعاء بمقدار {OVERLAP_DELTA:+.3f}، وهو أكبر من قيمة سؤال واحد "
      f"({1 / len(ANSWERABLE):.3f}) على {len(ANSWERABLE)} سؤالًا. والمفترض في هذه المُدوّنة أن "
      f"القطعين الثابتين لا يتمايزان عند هذا الحجم — والحركة الكبيرة تعني أن التقطيع غيّر شيئًا "
      f"غير الحدود")

check(len(UNANSWERED) >= 1,
      f"at least one answerable question must stay below full recall under every strategy — found "
      f"{len(UNANSWERED)}. This is a property of the corpus, not a bug: its answer lives in two "
      f"documents that share no vocabulary, and no chunk size reaches both in three chunks",
      f"يجب أن يبقى سؤال واحد على الأقل دون الاستدعاء التامّ عند كل الطرق — والعدد "
      f"{len(UNANSWERED)}. وهذه صفة في المُدوّنة لا خلل: إجابته في وثيقتين لا تشتركان في مفردة، ولا "
      f"يبلغ أي حجم مقطع كلتيهما في ثلاثة مقاطع")

report()

## What's next

**Tomorrow: the index.** Today you scored every chunk against every question with a full matrix
product, which is fine for 460 chunks and impossible for two million. D2 builds the same search as
a vector index — FAISS and Chroma — measures how much recall an approximate index gives up for its
speed, and persists a store that Wednesday's assistant and Friday's search both open.

It reads `chunk_comparison.parquet` to find out which strategy won here, so the store it builds is
the one your own measurement chose.

<div dir="rtl" align="right">

## ما التالي

**غدًا: الفهرس.** اليوم قِست كل مقطع مقابل كل سؤال بضرب مصفوفة كامل، وهو يصلح لأربعمئة وستين مقطعًا
ويستحيل لمليونين. ويبني اليوم الثاني البحث نفسه فهرسَ متّجهات — FAISS وChroma — ويقيس كم استدعاءً
يتنازل عنه الفهرس التقريبي في مقابل سرعته، ويحفظ مخزنًا يفتحه مساعد الأربعاء وبحث الجمعة كلاهما.

وهو يقرأ `chunk_comparison.parquet` ليعرف أي طريقة فازت هنا، فيكون المخزن الذي يبنيه هو ما اختاره
قياسك أنت.

</div>